In [1]:
import numpy as np

## Ridge Regression: 
####       it has closed form solution,
####       The solution is obtained by minimizing 
###        RSS(weights,alpha)= MSE+ alpha*(sum of squared weights)

In [2]:
class Ridge:
    def __init__(self,alpha):
        self.weights=None
        self.alpha=alpha
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def fit(self,X,y):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        a=X.T@X+self.alpha*np.identity(X.shape[1])
        b=X.T@y
        self.weights=np.linalg.inv(a)@b
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return float(mse)
    def __repr__(self,):
        return f"Ridge Regressor :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "

In [3]:
model=Ridge(3)
X=np.array(
    [[2,4,6],
    [3,6,9],
    [4,8,12]]
)
y=np.array([2,3,4])
model.fit(X,y)

In [4]:
model.predict(np.array([[5,10,15]]))

array([4.95227273])

In [5]:
lam=np.arange(0.25,10,0.25)
best_model=None
best_mse=float('inf')
for l in lam:
    model=Ridge(l)
    model.fit(X,y)
    y_hat=model.predict(X)
    mse=model.MSE(y,y_hat)
    print(model)
    if mse<best_mse:
        best_model=model
        best_mse=mse
print("*"*60)
print(f"Best model is \n {best_model}")
print("*"*60)

Ridge Regressor :
 -> Alpha = 0.25
 -> Weights = [0.01207648 0.07111707 0.14223415 0.21335122] 
Ridge Regressor :
 -> Alpha = 0.5
 -> Weights = [0.01558442 0.07099567 0.14199134 0.21298701] 
Ridge Regressor :
 -> Alpha = 0.75
 -> Weights = [0.01724964 0.07091519 0.14183038 0.21274557] 
Ridge Regressor :
 -> Alpha = 1.0
 -> Weights = [0.01821862 0.0708502  0.1417004  0.21255061] 
Ridge Regressor :
 -> Alpha = 1.25
 -> Weights = [0.01885014 0.07079275 0.14158551 0.21237826] 
Ridge Regressor :
 -> Alpha = 1.5
 -> Weights = [0.0192926  0.07073955 0.1414791  0.21221865] 
Ridge Regressor :
 -> Alpha = 1.75
 -> Weights = [0.01961853 0.07068898 0.14137797 0.21206695] 
Ridge Regressor :
 -> Alpha = 2.0
 -> Weights = [0.01986755 0.07064018 0.14128035 0.21192053] 
Ridge Regressor :
 -> Alpha = 2.25
 -> Weights = [0.02006316 0.07059261 0.14118521 0.21177782] 
Ridge Regressor :
 -> Alpha = 2.5
 -> Weights = [0.02022018 0.07054594 0.14109189 0.21163783] 
Ridge Regressor :
 -> Alpha = 2.75
 -> Weight

## LASSO Regression :
#### Lasso doesn't have any closed form solutio.
#### We continously reweight such that it minimizes the loss function.
#### For lasso the loss function is
###   RSS(weights,alpha)=MSE+alpha*(sum of absolute values of weights)
###   $$J(w) = \text{MSE} + \alpha \sum_{j=1}^{d} |w_j|$$
#### Lasso can converge a weight to zero, that means it can also identify the likely unnecessary variables

## Approach 1: Subgradient Descent

### Mathematical Overview
### Because $|w|$ has a sharp corner at zero, we use the subgradient of the penalty term, which translates directly to the element-wise sign of the weights:
## $$\nabla \Omega(w) = \alpha \cdot \text{sign}(w)$$

### Implementation

In [6]:
class LassoSubgradientDescent:
    def __init__(self,alpha):
        self.alpha=alpha
        self.weights=None
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return mse
    def fit(self,X,y,max_iter,learning_rate):
        self.weights=np.random.rand(X.shape[1]+1)
        
        X_expanded=X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        for _ in range(max_iter):
            cost=self.MSE(y,y_hat)+self.alpha*np.sum(np.abs(self.weights))
            p=X_expanded@self.weights
            w_grad_mse=(-2*X_expanded.T@(y-p))/len(y)
            w_grad_penality=self.alpha*np.sign(self.weights)
            grad=w_grad_mse+w_grad_penality
            self.weights-=learning_rate*grad
            print(f"Iternation {_+1} | Loss = {cost}")
            
    def __repr__(self,):
        return f"Lasso Regressor(SubgradientDescent) :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "

In [7]:
X_test = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 5.0],
    [3.0, 5.0, 1.0],
    [4.0, 1.0, 2.0]
])

# True relationship: y = 5 + 2*x1 + 0*x2 - 3*x3
y_test = np.array([0.0, 0.0, 8.0, 7.0]) 
model = LassoSubgradientDescent(alpha=0.1)
model.fit(X_test, y_test, max_iter=30, learning_rate=0.0001)

Iternation 1 | Loss = 29.45430973244344
Iternation 2 | Loss = 29.453994950033554
Iternation 3 | Loss = 29.45368179265279
Iternation 4 | Loss = 29.45337025210457
Iternation 5 | Loss = 29.453060320233554
Iternation 6 | Loss = 29.452751988925453
Iternation 7 | Loss = 29.452445250106802
Iternation 8 | Loss = 29.45214009574478
Iternation 9 | Loss = 29.451836517846985
Iternation 10 | Loss = 29.451534508461233
Iternation 11 | Loss = 29.451234059675365
Iternation 12 | Loss = 29.45093516361704
Iternation 13 | Loss = 29.450637812453532
Iternation 14 | Loss = 29.45034199839154
Iternation 15 | Loss = 29.45004771367698
Iternation 16 | Loss = 29.449754950594784
Iternation 17 | Loss = 29.449463701468726
Iternation 18 | Loss = 29.449173958661195
Iternation 19 | Loss = 29.44888571457303
Iternation 20 | Loss = 29.44859896164331
Iternation 21 | Loss = 29.448313692349156
Iternation 22 | Loss = 29.448029899205565
Iternation 23 | Loss = 29.44774757476519
Iternation 24 | Loss = 29.447466711618173
Iternation 

In [8]:
print(model)

Lasso Regressor(SubgradientDescent) :
 -> Alpha = 0.1
 -> Weights = [0.32468499 0.8707326  0.06298775 0.82744875] 


In [9]:
lam=np.arange(0.25,3,0.25)
best_model=None
best_cost=float('inf')
for l in lam:
    model=LassoSubgradientDescent(l)
    model.fit(X,y,max_iter=30,learning_rate=0.001)
    print(model)
    y_hat=model.predict(X)
    cost=model.MSE(y,y_hat)+model.alpha*np.sum(np.abs(model.weights))
    if cost<best_cost:
        best_model=model
        best_cost=cost
print("*"*60)
print(f"Best model is \n {best_model}")
print(f"Cost value of best_model is {best_cost}")
print("*"*60)

Iternation 1 | Loss = 0.515382980662654
Iternation 2 | Loss = 0.4422220856228612
Iternation 3 | Loss = 0.39469501175757815
Iternation 4 | Loss = 0.38033060982772054
Iternation 5 | Loss = 0.3698135302841177
Iternation 6 | Loss = 0.3620952664937896
Iternation 7 | Loss = 0.3564130609989796
Iternation 8 | Loss = 0.3522120303862914
Iternation 9 | Loss = 0.34908851344020475
Iternation 10 | Loss = 0.3467488586061026
Iternation 11 | Loss = 0.3449794430927498
Iternation 12 | Loss = 0.34362486265913583
Iternation 13 | Loss = 0.34257206533169776
Iternation 14 | Loss = 0.3417388091544878
Iternation 15 | Loss = 0.34106526554497735
Iternation 16 | Loss = 0.3405079109846042
Iternation 17 | Loss = 0.34003508340493177
Iternation 18 | Loss = 0.3396237495904592
Iternation 19 | Loss = 0.33925715356010494
Iternation 20 | Loss = 0.3389231058345285
Iternation 21 | Loss = 0.33861273892887894
Iternation 22 | Loss = 0.33831960201070466
Iternation 23 | Loss = 0.3380390022904518
Iternation 24 | Loss = 0.337767525

## Approach 2: Coordinate Descent (Analytical Selection)

### Mathematical Overview
#### Instead of updating all weights simultaneously, Coordinate Descent optimizes **one weight at a time** while holding all others constant. It calculates the partial residual $\rho_j$ (the error left over excluding feature $j$) and applies the **Soft-Thresholding Operator**:

## $$S(\rho_j, \alpha) = \text{sign}(\rho_j) \max(|\rho_j| - \alpha, 0)$$

#### This forces uninformative feature weights to snap cleanly to exactly `0.0`.

### Update rule:
$$
\beta_k =
\begin{cases} 
\frac{\sum_{i=1}^n r_i^{(k)} x_{ik} - \lambda}{\sum_{i=1}^n x_{ik}^2} & \text{if } \sum_{i=1}^n r_i^{(k)} x_{ik} > \lambda \\ 
\frac{\sum_{i=1}^n r_i^{(k)} x_{ik} + \lambda}{\sum_{i=1}^n x_{ik}^2} & \text{if } \sum_{i=1}^n r_i^{(k)} x_{ik} < -\lambda \\ 
0 & \text{if } -\lambda \le \sum_{i=1}^n r_i^{(k)} x_{ik} \le \lambda 
\end{cases}
$$

#### where $$ r_i^{(k)} = y_i - \sum_{j \neq k} x_{ij}\beta_j $$
$$r_i^{(k)} = (y_i - \hat{y}_i) + x_{ik}\beta_k$$

### Implementation

In [10]:
class LassoCoordinateDescent:
    def __init__(self,alpha):
        self.alpha=alpha
        self.weights=None
    def predict(self,X):
        X=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        prediction=X@self.weights
        return prediction
    def MSE(self,y,y_hat):
        mse=0
        for r,s in zip(y,y_hat):
            mse+=(r-s)**2
        return mse
    def r(self, X_expanded, y, y_hat, k):
        return (y - y_hat) + X_expanded[:, k] * self.weights[k]
    def update_rule(self,X,y,k):
        rtimesx=0
        x_expanded=np.concatenate((np.ones((X.shape[0],1)),X),axis=1)
        y_hat=x_expanded@self.weights
        for r,s in zip(self.r(x_expanded,y,y_hat,k+1),x_expanded[:,k+1]):
            rtimesx+=r*s
        if rtimesx>self.alpha:
            update_value=(rtimesx-self.alpha)/np.sum(x_expanded[:,k+1]**2)
        elif rtimesx<-self.alpha:
            update_value=(rtimesx+self.alpha)/np.sum(x_expanded[:,k+1]**2)
        else:
            update_value=0
        return update_value
            
    def fit(self,X,y,max_iter):
        samples,features=X.shape
        self.weights=np.random.rand(features+1)
        for _ in range(max_iter):
            y_hat_features = X @ self.weights[1:]
            self.weights[0] = np.mean(y - y_hat_features)
            for k in range(features):
                update_value=self.update_rule(X,y,k)
                self.weights[k+1]=update_value
            y_pred=self.predict(X)
            print(f"Iteration : {_+1} | Loss : {self.MSE(y,y_pred)}")
    def __repr__(self,):
        return f"Lasso Regressor(Coordinate Descent) :\n -> Alpha = {self.alpha}\n -> Weights = {self.weights} "
            
        
        

In [11]:
X_test = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 5.0],
    [3.0, 5.0, 1.0],
    [4.0, 1.0, 2.0]
])

# True relationship: y = 5 + 2*x1 + 0*x2 - 3*x3
y_test = np.array([0.0, 0.0, 8.0, 7.0]) 
model = LassoCoordinateDescent(alpha=0.1)
model.fit(X_test, y_test, max_iter=30)

Iteration : 1 | Loss : 40.661035048024466
Iteration : 2 | Loss : 20.088168939120507
Iteration : 3 | Loss : 9.835848305642088
Iteration : 4 | Loss : 4.864347233010137
Iteration : 5 | Loss : 2.511702309204231
Iteration : 6 | Loss : 1.4152279307322124
Iteration : 7 | Loss : 0.9010199409133517
Iteration : 8 | Loss : 0.6481508455597456
Iteration : 9 | Loss : 0.5099588915618942
Iteration : 10 | Loss : 0.4223378193204101
Iteration : 11 | Loss : 0.3585103329405767
Iteration : 12 | Loss : 0.3076908155481984
Iteration : 13 | Loss : 0.26551835145098907
Iteration : 14 | Loss : 0.2300233476879857
Iteration : 15 | Loss : 0.20005774872354992
Iteration : 16 | Loss : 0.17475029462725356
Iteration : 17 | Loss : 0.15335041649361664
Iteration : 18 | Loss : 0.13519894655341222
Iteration : 19 | Loss : 0.11972852051025414
Iteration : 20 | Loss : 0.10646361549664815
Iteration : 21 | Loss : 0.0950145799081164
Iteration : 22 | Loss : 0.08506723157963056
Iteration : 23 | Loss : 0.07637067098750225
Iteration : 24

In [12]:
y_pred=model.predict(X_test)
print(model.MSE(y_test,y_pred))
print(model)


0.037864005390827994
Lasso Regressor(Coordinate Descent) :
 -> Alpha = 0.1
 -> Weights = [ 1.887058    1.91409639  0.36466275 -1.45204672] 


In [13]:
lam=np.arange(0.25,3,0.25)
best_model=None
best_cost=float('inf')
for l in lam:
    model=LassoCoordinateDescent(l)
    model.fit(X,y,max_iter=25)
    print(model)
    y_hat=model.predict(X)
    cost=model.MSE(y,y_hat)+model.alpha*np.sum(np.abs(model.weights))
    if cost<best_cost:
        best_model=model
        best_cost=cost
print("*"*60)
print(f"Best model is \n {best_model}")
print(f"Cost value of best_model is {best_cost}")
print("*"*60)

Iteration : 1 | Loss : 1.8970166633682481
Iteration : 2 | Loss : 1.634373192909675
Iteration : 3 | Loss : 1.4074293556687025
Iteration : 4 | Loss : 1.2113810574069424
Iteration : 5 | Loss : 1.0420674228271067
Iteration : 6 | Loss : 0.8958848981491617
Iteration : 7 | Loss : 0.7697128093942471
Iteration : 8 | Loss : 0.6608488496382289
Iteration : 9 | Loss : 0.5669531718974897
Iteration : 10 | Loss : 0.4859999406170664
Iteration : 11 | Loss : 0.4162353475540873
Iteration : 12 | Loss : 0.35614123031545664
Iteration : 13 | Loss : 0.3044035466293346
Iteration : 14 | Loss : 0.2598850569552538
Iteration : 15 | Loss : 0.2216016543050786
Iteration : 16 | Loss : 0.1887018549221614
Iteration : 17 | Loss : 0.16044902827987476
Iteration : 18 | Loss : 0.1362060010398461
Iteration : 19 | Loss : 0.115421718304981
Iteration : 20 | Loss : 0.097619687709512
Iteration : 21 | Loss : 0.08238796847216048
Iteration : 22 | Loss : 0.06937049924797044
Iteration : 23 | Loss : 0.05825958609886616
Iteration : 24 | L